In [1]:
# Setup
import pandas as pd

import os
import dotenv
dotenv.load_dotenv()

username = os.getenv('USER')
pwd = os.getenv('PWD')
host = os.getenv('HOST')
port = os.getenv('PORT')
default_db = os.getenv('DEFAULT_DB')

import psycopg
# import psycopg2.extras as extras

from IPython.display import display, HTML

# import matplotlib.pyplot as plt
# import seaborn as sns

In [12]:
# Functions
def display_scrollable(df, max_rows=999):
    pd.options.display.max_rows = 999
    pd.options.display.max_columns = 999
    html = df.to_html(max_rows=max_rows)
    html = f'<div style="max-height: 500px; overflow-y: scroll;">{html}</div>'
    display(HTML(html))
    pd.options.display.max_rows = 15
    pd.options.display.max_columns = 15

def select_table(query: str) -> pd.DataFrame:
    conn = psycopg.connect(
        host = host,
        dbname = default_db,
        user = username,
        password = pwd,
        port = 5432
    )
    with conn:
        with conn.cursor() as cur:
            cur.execute(query)
            column_headers = [desc[0] for desc in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=column_headers)
            conn.commit()
            cur.close()
    return df


In [5]:
create_table_query = '''
CREATE TABLE IF NOT EXISTS users (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    age INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
'''

insert_data_query = '''
INSERT INTO users (name, email, age)
VALUES
    ('Alice', 'alice@example.com', 30),
    ('Bob', 'bob@example.com', 25),
    ('Charlie', 'charlie@example.com', 35)
ON CONFLICT (email) DO NOTHING; -- Prevent duplicate insertions
'''


In [6]:
conn = psycopg.connect(
    host = host,
    dbname = default_db,
    user = username,
    password = pwd,
    port = 5432
)


with conn:
    with conn.cursor() as cur:
        # Create the table
        cur.execute(create_table_query)
        print("Table 'users' created (if not already exists).")
        
        # Insert sample data
        cur.execute(insert_data_query)
        print("Sample data inserted into 'users' table.")

conn.close()
        

Table 'users' created (if not already exists).
Sample data inserted into 'users' table.


In [13]:

select_query = f'''
SELECT * 
FROM users
'''

df_c = select_table(select_query)

display_scrollable(df_c)

,id,name,email,age,created_at
0,1,Alice,alice@example.com,30,2025-01-03 15:29:17.532793
1,2,Bob,bob@example.com,25,2025-01-03 15:29:17.532793
2,3,Charlie,charlie@example.com,35,2025-01-03 15:29:17.532793


In [17]:
select_tables = '''
    SELECT * FROM pg_catalog.pg_tables
    where schemaname = 'public';
    '''

df_c = select_table(select_tables)
display_scrollable(df_c)

,schemaname,tablename,tableowner,tablespace,hasindexes,hasrules,hastriggers,rowsecurity
0,public,users,postgres,None,True,False,False,False
